In [1]:
import sys
import importlib
import types

# Check if imp is missing (it will be on Python 3.12+)
if "imp" not in sys.modules:
    # Create a fake "imp" module
    fake_imp = types.ModuleType("imp")
    # Map imp.reload to importlib.reload
    fake_imp.reload = importlib.reload
    # Inject it into system modules
    sys.modules["imp"] = fake_imp
    print("Patched 'imp' module for Python 3.12 compatibility.")

# Now this should work without crashing
%load_ext autoreload
%autoreload 2

Patched 'imp' module for Python 3.12 compatibility.


In [2]:
!git clone https://github.com/mattias32199/Kalman-Filter-RL-POMDP.git
%cd Kalman-Filter-RL-POMDP/

Cloning into 'Kalman-Filter-RL-POMDP'...
remote: Enumerating objects: 177, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 177 (delta 116), reused 110 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (177/177), 47.24 KiB | 5.91 MiB/s, done.
Resolving deltas: 100% (116/116), done.
/content/Kalman-Filter-RL-POMDP


In [3]:
!git pull

Already up to date.


In [9]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from src.train_pendulum import train_separate
from src.util import set_seed, save_data

save_path = "/content/drive/MyDrive/Autonomous Control Systems/baseline"

def pipe(seed=42, noise=0.0, save_path=""):

    set_seed(seed)

    agent, rewards, eval_rewards = train_separate(
        num_episodes=500,
        noise_std=noise,       # set to 0.1 for noisy variant
        device=device
    )

    if noise == 0.05:
        noise = 's'
    elif noise == 0.1:
        noise = 'm'
    elif noise == 0.3:
        noise = 'l'
    elif noise == 0:
        noise = 'na'

    save_data(
        {},
        seed=seed,
        group="separate",
        policy="td3",
        rewards=rewards,
        evals=eval_rewards,
        agent=agent,
        path=save_path,
        noise=noise,

    )

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/'Autonomous Control Systems'/Results

In [ ]:
seeds = [42, 123, 256, 789, 1024]
env_noise = [0.05, 0.1, 0.3]
env_noise = [0.1, 0.05, 0.3]


for seed in seeds:
    for noise in env_noise:
        print(f"seed: {seed}, noise: {noise}")
        pipe(seed, noise, save_path)


seed: 42, noise: 0.1
Episode   10 | Avg Reward: -1150.79 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   20 | Avg Reward: -1509.95 | Est Loss: 2.6793 | Q diag: [0.0075, 0.3557] | R diag: [0.0397, 0.0781]
  → Eval (10 eps): -1239.77
Episode   30 | Avg Reward: -1384.35 | Est Loss: 2.2829 | Q diag: [0.0066, 0.7835] | R diag: [0.0160, 0.0145]
Episode   40 | Avg Reward: -1346.64 | Est Loss: 2.3241 | Q diag: [0.0090, 0.9096] | R diag: [0.0084, 0.0103]
Episode   50 | Avg Reward: -1274.29 | Est Loss: 2.0405 | Q diag: [0.0078, 0.6562] | R diag: [0.0055, 0.0067]
  → Eval (10 eps): -1272.58
Episode   60 | Avg Reward: -1228.91 | Est Loss: 1.9722 | Q diag: [0.0019, 0.1021] | R diag: [0.0020, 0.0023]
Episode   70 | Avg Reward: -1366.60 | Est Loss: 1.9507 | Q diag: [0.0005, 0.0040] | R diag: [0.0007, 0.0009]
  → Eval (10 eps): -1322.02
Episode   80 | Avg Reward: -1274.87 | Est Loss: 1.7920 | Q diag: [0.0003, 0.0011] | R diag: [0.0006, 0.0007]
Episode   90 | Avg Reward:

In [ ]:
# 3-14 #2
from src.train import train_separate

agent, rewards, eval_rewards = train_separate(
    num_episodes=500,
    noise_std=0.0,       # set to 0.1 for noisy variant
    device=device
)

# Print final learned EKF parameters
print("\n" + "=" * 50)
print("Final learned EKF parameters:")
print(f"Q (process noise):\n{agent.ekf.Q.detach().cpu().numpy()}")
print(f"R (measurement noise):\n{agent.ekf.R.detach().cpu().numpy()}")

Episode   10 | Avg Reward: -1205.88 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   20 | Avg Reward: -1403.10 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1343.86
Episode   30 | Avg Reward: -1356.07 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   40 | Avg Reward: -1309.21 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   50 | Avg Reward: -1373.34 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1417.31
Episode   60 | Avg Reward: -1324.19 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   70 | Avg Reward: -1329.05 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1306.04
Episode   80 | Avg Reward: -1392.93 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   90 | Avg Reward: -1488.02 | Est Loss: nan | Q diag: [0.018

In [ ]:
# 3-14 #1
from src.train import train_separate

agent, rewards, eval_rewards = train_separate(
    num_episodes=500,
    noise_std=0.0,       # set to 0.1 for noisy variant
    device=device
)

# Print final learned EKF parameters
print("\n" + "=" * 50)
print("Final learned EKF parameters:")
print(f"Q (process noise):\n{agent.ekf.Q.detach().cpu().numpy()}")
print(f"R (measurement noise):\n{agent.ekf.R.detach().cpu().numpy()}")

Episode   10 | Avg Reward: -1232.35 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   20 | Avg Reward: -1218.97 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1440.40
Episode   30 | Avg Reward: -1352.81 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   40 | Avg Reward: -1340.66 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   50 | Avg Reward: -1253.89 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1248.66
Episode   60 | Avg Reward: -1351.37 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   70 | Avg Reward: -1391.06 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1280.26
Episode   80 | Avg Reward: -1262.56 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   90 | Avg Reward: -1371.03 | Est Loss: nan | Q diag: [0.018

In [ ]:
# 3-14 0.05
from src.train import train_separate

agent, rewards, eval_rewards = train_separate(
    num_episodes=500,
    noise_std=0.05,       # set to 0.1 for noisy variant
    device=device
)

# Print final learned EKF parameters
print("\n" + "=" * 50)
print("Final learned EKF parameters:")
print(f"Q (process noise):\n{agent.ekf.Q.detach().cpu().numpy()}")
print(f"R (measurement noise):\n{agent.ekf.R.detach().cpu().numpy()}")

Episode   10 | Avg Reward: -1205.27 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   20 | Avg Reward: -1232.54 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1313.11
Episode   30 | Avg Reward: -1241.52 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   40 | Avg Reward: -1264.12 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   50 | Avg Reward: -1318.86 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1297.41
Episode   60 | Avg Reward: -1370.46 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   70 | Avg Reward: -1223.07 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1303.26
Episode   80 | Avg Reward: -1361.90 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   90 | Avg Reward: -1335.68 | Est Loss: nan | Q diag: [0.018

In [ ]:
# 3-14 0.1
from src.train import train_separate

agent, rewards, eval_rewards = train_separate(
    num_episodes=500,
    noise_std=0.1,       # set to 0.1 for noisy variant
    device=device
)

# Print final learned EKF parameters
print("\n" + "=" * 50)
print("Final learned EKF parameters:")
print(f"Q (process noise):\n{agent.ekf.Q.detach().cpu().numpy()}")
print(f"R (measurement noise):\n{agent.ekf.R.detach().cpu().numpy()}")

Episode   10 | Avg Reward: -1181.81 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   20 | Avg Reward: -1298.03 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1271.92
Episode   30 | Avg Reward: -1365.87 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   40 | Avg Reward: -1392.65 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   50 | Avg Reward: -1269.14 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1419.72
Episode   60 | Avg Reward: -1321.11 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   70 | Avg Reward: -1405.50 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1260.86
Episode   80 | Avg Reward: -1356.28 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   90 | Avg Reward: -1301.32 | Est Loss: nan | Q diag: [0.018

In [ ]:
# 3-14 0.3
from src.train import train_separate

agent, rewards, eval_rewards = train_separate(
    num_episodes=500,
    noise_std=0.3,       # set to 0.1 for noisy variant
    device=device
)

# Print final learned EKF parameters
print("\n" + "=" * 50)
print("Final learned EKF parameters:")
print(f"Q (process noise):\n{agent.ekf.Q.detach().cpu().numpy()}")
print(f"R (measurement noise):\n{agent.ekf.R.detach().cpu().numpy()}")

Episode   10 | Avg Reward: -1160.80 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   20 | Avg Reward: -1266.61 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1287.66
Episode   30 | Avg Reward: -1219.45 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   40 | Avg Reward: -1215.64 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   50 | Avg Reward: -1289.67 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1355.71
Episode   60 | Avg Reward: -1305.55 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   70 | Avg Reward: -1396.78 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1437.48
Episode   80 | Avg Reward: -1180.37 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   90 | Avg Reward: -1236.10 | Est Loss: nan | Q diag: [0.018

In [ ]:
# 3-14
from src.train import train_separate

agent, rewards, eval_rewards = train_separate(
    num_episodes=500,
    noise_std=0.0,       # set to 0.1 for noisy variant
    device=device
)

# Print final learned EKF parameters
print("\n" + "=" * 50)
print("Final learned EKF parameters:")
print(f"Q (process noise):\n{agent.ekf.Q.detach().cpu().numpy()}")
print(f"R (measurement noise):\n{agent.ekf.R.detach().cpu().numpy()}")

Episode   10 | Avg Reward:  -936.77 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   20 | Avg Reward: -1381.66 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1277.43
Episode   30 | Avg Reward: -1339.01 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   40 | Avg Reward: -1281.45 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   50 | Avg Reward: -1401.91 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1300.02
Episode   60 | Avg Reward: -1295.15 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   70 | Avg Reward: -1221.28 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
  → Eval (10 eps): -1319.19
Episode   80 | Avg Reward: -1398.65 | Est Loss: nan | Q diag: [0.0183, 0.0183] | R diag: [0.1353, 0.1353]
Episode   90 | Avg Reward: -1338.41 | Est Loss: nan | Q diag: [0.018

In [ ]:
"""
Partially Observable Lunar Lander Environment
=============================================

Wraps LunarLander-v3 to hide velocity components, mirroring the
PartiallyObservablePendulum pattern.

Full state (8D):
    [x, y, vx, vy, angle, angular_vel, left_leg_contact, right_leg_contact]

Masked observation (5D, default):
    [x, y, angle, left_leg_contact, right_leg_contact]

Hidden (3D):  vx, vy, angular_vel  — these are what the EKF must estimate.

Design notes for EKF integration:
    - The continuous state for the EKF is 6D: [x, y, vx, vy, angle, angular_vel]
    - The observation model maps this 6D state to 3D: [x, y, angle]
    - Leg contacts are binary/discrete and passed through unchanged —
      they should be appended to the EKF estimate before feeding into the policy.
    - `info["full_state"]`   → full 8D vector (for logging / ground truth comparison)
    - `info["continuous_state"]` → 6D continuous-only (for EKF supervision if needed)
    - `info["leg_contacts"]`    → 2D binary (for pass-through)
"""

import gymnasium as gym
import numpy as np


# Indices into the full 8D LunarLander observation
IDX_X        = 0
IDX_Y        = 1
IDX_VX       = 2
IDX_VY       = 3
IDX_ANGLE    = 4
IDX_ANG_VEL  = 5
IDX_LEFT_LEG = 6
IDX_RIGHT_LEG = 7

# Default: hide all three velocity components
DEFAULT_HIDDEN = [IDX_VX, IDX_VY, IDX_ANG_VEL]

# Continuous vs discrete index sets
CONTINUOUS_IDX = [IDX_X, IDX_Y, IDX_VX, IDX_VY, IDX_ANGLE, IDX_ANG_VEL]
LEG_CONTACT_IDX = [IDX_LEFT_LEG, IDX_RIGHT_LEG]


class PartiallyObservableLunarLander(gym.Wrapper):
    """
    Wraps LunarLander-v3 to produce partial observations by dropping
    specified state components (default: velocities).

    Parameters
    ----------
    hidden_indices : list[int]
        Indices into the 8D state vector to hide. Must be a subset of
        the continuous indices [0..5]; leg contacts [6,7] are always
        included in the observation (they're discrete pass-through).
    noise_std : float
        Gaussian noise std added to the *continuous* observed components.
        Leg contacts are never noised.
    """

    def __init__(self, hidden_indices=None, noise_std=0.0):
        super().__init__(gym.make("LunarLander-v3"))

        self.hidden_indices = hidden_indices or DEFAULT_HIDDEN
        self.noise_std = noise_std

        # Validate: only continuous indices can be hidden
        for idx in self.hidden_indices:
            assert idx in CONTINUOUS_IDX, (
                f"Can only hide continuous state indices {CONTINUOUS_IDX}, got {idx}"
            )

        # Observed continuous indices = continuous minus hidden
        self.observed_continuous_idx = [
            i for i in CONTINUOUS_IDX if i not in self.hidden_indices
        ]
        # Full observed = observed continuous + leg contacts
        self.observed_idx = self.observed_continuous_idx + LEG_CONTACT_IDX

        obs_dim = len(self.observed_idx)
        # Build observation bounds from the original space
        orig_low = self.env.observation_space.low
        orig_high = self.env.observation_space.high
        self.observation_space = gym.spaces.Box(
            low=orig_low[self.observed_idx],
            high=orig_high[self.observed_idx],
            shape=(obs_dim,),
            dtype=np.float32,
        )

    def _mask(self, obs):
        """Extract observed components and optionally add noise."""
        masked = obs[self.observed_idx].copy()

        if self.noise_std > 0:
            # Noise only the continuous part, not leg contacts
            n_cont = len(self.observed_continuous_idx)
            masked[:n_cont] += np.random.normal(
                0, self.noise_std, size=n_cont
            )

        return masked.astype(np.float32)

    def _enrich_info(self, obs, info):
        """Attach full state, continuous state, and leg contacts to info."""
        info["full_state"] = obs                        # 8D
        info["continuous_state"] = obs[CONTINUOUS_IDX]   # 6D — for EKF ground truth
        info["leg_contacts"] = obs[LEG_CONTACT_IDX]      # 2D — binary pass-through
        return info

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        info = self._enrich_info(obs, info)
        return self._mask(obs), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        info = self._enrich_info(obs, info)
        return self._mask(obs), reward, terminated, truncated, info


# ---------------------------------------------------------------------------
# Convenience constructors matching the Pendulum naming convention
# ---------------------------------------------------------------------------

def make_clean_lunar_lander():
    """Position + angle only, no noise."""
    return PartiallyObservableLunarLander(noise_std=0.0)


def make_noisy_lunar_lander(noise_std=0.05):
    """Position + angle only, with Gaussian sensor noise."""
    return PartiallyObservableLunarLander(noise_std=noise_std)


if __name__ == "__main__":
    print("=" * 60)
    print("Partially Observable Lunar Lander — smoke test")
    print("=" * 60)

    # --- Clean variant ---
    env = make_clean_lunar_lander()
    obs, info = env.reset(seed=42)
    print(f"\n[Clean]")
    print(f"  Masked obs shape : {obs.shape}")              # (5,)
    print(f"  Masked obs       : {obs}")
    print(f"  Full state shape : {info['full_state'].shape}")  # (8,)
    print(f"  Continuous state : {info['continuous_state'].shape}")  # (6,)
    print(f"  Leg contacts     : {info['leg_contacts']}")

    # Take a few steps
    total_reward = 0.0
    for _ in range(50):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break
    print(f"  After 50 steps   : obs={obs}, reward_sum={total_reward:.1f}")
    env.close()

    # --- Noisy variant ---
    env_noisy = make_noisy_lunar_lander(noise_std=0.1)
    obs, info = env_noisy.reset(seed=42)
    print(f"\n[Noisy, std=0.1]")
    print(f"  Masked obs shape : {obs.shape}")
    print(f"  Masked obs       : {obs}")
    env_noisy.close()

    # --- Custom: hide only vx, vy (keep angular velocity) ---
    env_custom = PartiallyObservableLunarLander(
        hidden_indices=[IDX_VX, IDX_VY], noise_std=0.0
    )
    obs, info = env_custom.reset(seed=42)
    print(f"\n[Custom: hide vx,vy only]")
    print(f"  Masked obs shape : {obs.shape}")              # (6,)
    print(f"  Observed indices : {env_custom.observed_idx}")
    env_custom.close()

    print("\nAll checks passed.")